# 01 — Préparation des données (source : PostgreSQL)

Construction du jeu de données client (grain = 1 ligne par client) **par requête SQL
directe** sur l'entrepôt PostgreSQL `churn_dw` (tables `fact_compte_client`,
`dim_client`, `dim_industry`), construit par le pipeline ETL (`01_etl/`).

Aucun fichier CSV brut n'est lu ici : toute l'agrégation compte -> client se fait
côté SQL (`GROUP BY customer_no`), voir `src/prepare_data.py` pour la requête complète.

Variables de connexion (`.env` ou variables d'environnement) :
```
PGHOST=localhost
PGPORT=5432
PGDATABASE=churn_dw
PGUSER=postgres
PGPASSWORD=postgres
```

In [1]:
import sys
sys.path.append('../src')
from prepare_data import build_client_dataset, check_connection
from db import get_engine

check_connection()
engine = get_engine()
df = build_client_dataset(engine)
df.to_csv('../data/client_churn_dataset.csv', index=False)
print(df.shape)
df.head()

[db] Connexion OK à 'churn_dw' sur localhost:5432 — 7 tables trouvées.
[prepare_data] Casse des colonnes détectée en base : upper (ex: 'CLIENT_SK')
[prepare_data] Exécution de la requête d'agrégation compte -> client sur PostgreSQL...
[prepare_data] 363,569 clients récupérés depuis l'entrepôt.
(363569, 32)


,customer_no,nationality,residence,marital_status,age,nature_client,partyclass,lob,score_kyc,completed_file,...,taux_fixe_moyen,nb_devises,nb_secteurs,nb_agences,a_credit,a_epargne_placement,a_compte_courant,a_coffre,flag_incoherence_cloture,secteur_principal
0,C000001,TN,TN,Non applicable,46,NaN,Corporate,30,H2,Non,...,0.0,3,1,1,0,0,0,0,0,Autre
1,C000002,TN,TN,Non applicable,46,NaN,Corporate,30,LR,Non,...,0.0,1,1,1,0,0,0,0,0,Autre
2,C000003,TN,TN,Non applicable,46,PM,Corporate,13,MR,Non,...,0.0,2,1,1,0,0,0,0,0,culture de cereales (exception du riz)
3,C000004,TN,TN,C,24,PPH,Retail,4,LR,YES,...,0.0,0,1,1,0,0,0,0,0,Other
4,C000005,TN,TN,M,49,PPH,Retail,4,LR,YES,...,0.0,1,1,1,0,1,0,0,0,Other


## Distribution de la cible (churn)

`churn` = `CLIENT_FULL_CHURN` : 1 si le client n'a plus aucun compte actif (tous ses comptes sont clôturés).

In [2]:
df['churn'].value_counts(normalize=True).rename('proportion')

churn
0    0.554973
1    0.445027
Name: proportion, dtype: float64

## Valeurs manquantes

In [3]:
df.isna().mean().sort_values(ascending=False).head(10)

anciennete_compte_moy_annees       0.382310
anciennete_compte_max_annees       0.382310
jours_depuis_derniere_revue_moy    0.075325
anciennete_client_annees           0.037264
score_kyc                          0.002283
nature_client                      0.000017
marital_status                     0.000000
customer_no                        0.000000
lob                                0.000000
partyclass                         0.000000
dtype: float64

## ⚠️ Fuites de données identifiées

Voir `comparison.md` pour le détail complet. En résumé :
- `nb_comptes_clos` est **exclue** : elle définit littéralement la cible (`churn = (nb_comptes_clos == nb_comptes)`).
- `nb_devises` est **exclue** : `nb_devises = 0` coïncide avec `churn = 1` dans ~100% des cas pour 38% des clients (artefact de complétude des données produit côté SI source, pas un vrai signal comportemental). Un test de sensibilité confirme une performance quasi identique sans cette variable.

## Sauvegarde (déjà faite ci-dessus, rappel du chemin)

In [4]:
print('Sauvegardé :', '../data/client_churn_dataset.csv', df.shape)

Sauvegardé : ../data/client_churn_dataset.csv (363569, 32)
